In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
# from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI


c:\micromamba\envs\langchain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash",)

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza maker quit his job?\n\nBecause he was tired of getting kneaded!',
 'explanation': 'The joke plays on the double meaning of the word "kneaded." Here\'s the breakdown:\n\n* **Literal Meaning (Pizza Making):** In the context of making pizza, "kneading" refers to the process of working the dough with your hands to develop the gluten and give it the right texture. This is a physically demanding task.\n\n* **Figurative Meaning (Tired/Stressed):** "Kneaded" sounds like "needed." The joke implies that the pizza maker felt overworked, stressed, and generally put-upon by the demands of his job. He was constantly being "needed" to make pizzas.\n\nThe humor comes from the pun. The pizza maker quit not just because he was physically tired from kneading dough, but also because he was emotionally "kneaded" (needed/stressed) and overwhelmed by the demands of his job.'}

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza maker quit his job?\n\nBecause he was tired of getting kneaded!', 'explanation': 'The joke plays on the double meaning of the word "kneaded." Here\'s the breakdown:\n\n* **Literal Meaning (Pizza Making):** In the context of making pizza, "kneading" refers to the process of working the dough with your hands to develop the gluten and give it the right texture. This is a physically demanding task.\n\n* **Figurative Meaning (Tired/Stressed):** "Kneaded" sounds like "needed." The joke implies that the pizza maker felt overworked, stressed, and generally put-upon by the demands of his job. He was constantly being "needed" to make pizzas.\n\nThe humor comes from the pun. The pizza maker quit not just because he was physically tired from kneading dough, but also because he was emotionally "kneaded" (needed/stressed) and overwhelmed by the demands of his job.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns'

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza maker quit his job?\n\nBecause he was tired of getting kneaded!', 'explanation': 'The joke plays on the double meaning of the word "kneaded." Here\'s the breakdown:\n\n* **Literal Meaning (Pizza Making):** In the context of making pizza, "kneading" refers to the process of working the dough with your hands to develop the gluten and give it the right texture. This is a physically demanding task.\n\n* **Figurative Meaning (Tired/Stressed):** "Kneaded" sounds like "needed." The joke implies that the pizza maker felt overworked, stressed, and generally put-upon by the demands of his job. He was constantly being "needed" to make pizzas.\n\nThe humor comes from the pun. The pizza maker quit not just because he was physically tired from kneading dough, but also because he was emotionally "kneaded" (needed/stressed) and overwhelmed by the demands of his job.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti blush?\n\nBecause it saw the meatball sauce!',
 'explanation': 'The joke relies on a double entendre, meaning it has two possible interpretations:\n\n*   **Literal interpretation:** Spaghetti is a type of pasta, and meatballs are often served with a tomato-based sauce. The joke anthropomorphizes the spaghetti, giving it human-like qualities like blushing.\n\n*   **Figurative interpretation:** "Meatball sauce" sounds similar to "meatball source." The joke plays on the idea that the spaghetti is blushing because it is embarrassed or flustered by the "source" of the meatball sauce, implying a risqué or suggestive reason.\n\nThe humor comes from the unexpected and slightly suggestive second meaning, which contrasts with the innocent image of pasta and meatballs.'}

In [12]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti blush?\n\nBecause it saw the meatball sauce!', 'explanation': 'The joke relies on a double entendre, meaning it has two possible interpretations:\n\n*   **Literal interpretation:** Spaghetti is a type of pasta, and meatballs are often served with a tomato-based sauce. The joke anthropomorphizes the spaghetti, giving it human-like qualities like blushing.\n\n*   **Figurative interpretation:** "Meatball sauce" sounds similar to "meatball source." The joke plays on the idea that the spaghetti is blushing because it is embarrassed or flustered by the "source" of the meatball sauce, implying a risqué or suggestive reason.\n\nThe humor comes from the unexpected and slightly suggestive second meaning, which contrasts with the innocent image of pasta and meatballs.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f09ecbc-b9d3-6d8a-8002-eee85d2c9a6a'}}, metadata={'source': 'loop'

In [13]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti blush?\n\nBecause it saw the meatball sauce!', 'explanation': 'The joke relies on a double entendre, meaning it has two possible interpretations:\n\n*   **Literal interpretation:** Spaghetti is a type of pasta, and meatballs are often served with a tomato-based sauce. The joke anthropomorphizes the spaghetti, giving it human-like qualities like blushing.\n\n*   **Figurative interpretation:** "Meatball sauce" sounds similar to "meatball source." The joke plays on the idea that the spaghetti is blushing because it is embarrassed or flustered by the "source" of the meatball sauce, implying a risqué or suggestive reason.\n\nThe humor comes from the unexpected and slightly suggestive second meaning, which contrasts with the innocent image of pasta and meatballs.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f09ecbc-b9d3-6d8a-8002-eee85d2c9a6a'}}, metadata={'source': 'loop